# Lab 10-02: Require Human Approval for High-Risk Actions

**Book chapter:** Chapter 10: Securing and Governing Agentic Workflows  
**Estimated time:** 20 minutes

## Overview

Cymbal Retail allows small refunds automatically, requires a store associate to
approve larger ones, and never allows a refund larger than the order total. In this
lab, you implement the policy with ADK tool confirmation and a before-tool callback,
approve and reject paused refunds, and review the audit trail.

## Learning Objectives

By the end of this lab, you will be able to:

- Require confirmation for a tool based on its arguments
- Present an approval request and resume the agent with a decision
- Enforce a hard limit deterministically with a callback
- Record approvals in an audit trail

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Control | Use it for |
|---|---|
| **Deterministic guardrail** (callback or gateway policy) | Rules that must never be broken |
| **Human approval** (tool confirmation) | Decisions that need judgment, such as large refunds |
| **Instruction** | Guidance only; a prompt can override it, so it is not a control |

## Setup

In [ ]:
%pip install --quiet --upgrade google-adk

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Define the refund tool, the approval rule, and a guardrail

The refund policy has three parts:

| Rule | Enforced by |
|---|---|
| Refunds up to USD 50 are automatic | `needs_approval` returns `False` |
| Larger refunds need an associate's approval | `FunctionTool(..., require_confirmation=needs_approval)` |
| Never refund more than the order total | `within_order_total`, a before-tool callback |

`record` writes each event to an audit trail.

In [ ]:
import datetime
import os
import uuid

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
ORDER_TOTALS = {"10421": 30.00, "10422": 450.00, "10423": 1200.00}
AUDIT = []


def record(event, **details):
    """Adds an entry to the audit trail."""
    now = datetime.datetime.now().strftime("%H:%M:%S")
    AUDIT.append({"time": now, "event": event, **details})


def issue_refund(order_id: str, amount: float) -> dict:
    """Issues a refund to the shopper for a Cymbal Retail order."""
    record("refund_issued", order_id=order_id, amount=amount)
    return {"status": "refunded", "order_id": order_id, "amount": amount}


def needs_approval(order_id: str, amount: float) -> bool:
    """Refunds above USD 50 need a store associate's approval."""
    return amount > 50


def within_order_total(tool, args, tool_context):
    """Deterministic guardrail: never refund more than the order total."""
    total = ORDER_TOTALS.get(args.get("order_id"), 0)
    if args.get("amount", 0) > total:
        record("refund_blocked", **args, reason="exceeds order total")
        return {"status": "blocked",
                "reason": f"The order total is only USD {total:.2f}."}
    return None

## Step 2: Build the agent and a helper that shows approval requests

When a refund needs approval, ADK pauses the agent and emits a function call named
`adk_request_confirmation`, which contains the original tool call. `send` collects those
requests, and `ask` prints them, just as an approval app would show them to an
associate. The first refund is below the threshold.

In [ ]:
agent = Agent(
    name="refund_agent", model=MODEL,
    instruction="You process refunds for Cymbal Retail. Call issue_refund "
                "with the order number and amount that the shopper asks "
                "for. Then tell the shopper the outcome in one sentence.",
    tools=[FunctionTool(issue_refund, require_confirmation=needs_approval)],
    before_tool_callback=within_order_total)
runner = Runner(app_name="refunds", agent=agent,
                session_service=InMemorySessionService(),
                auto_create_session=True)


def send(session_id, message):
    """Runs one turn and returns any approval requests."""
    requests = []
    for event in runner.run(user_id="shopper-1", session_id=session_id,
                            new_message=message):
        for call in event.get_function_calls():
            if call.name == "adk_request_confirmation":
                requests.append(call)
        text = event.content.parts[0].text if event.content else None
        if event.is_final_response() and text:
            print("   Agent:", text)
    return requests


def ask(text):
    """Sends a shopper message and prints any approval request."""
    print(f"Shopper: {text}")
    session_id = str(uuid.uuid4())
    message = types.Content(role="user", parts=[types.Part(text=text)])
    requests = send(session_id, message)
    for request in requests:
        call = request.args["originalFunctionCall"]
        print(f"   PAUSED: approval needed for {call['name']}"
              f"({call['args']})")
    return session_id, requests


ask("Please refund USD 20 for order 10421.")

**Expected result:** the USD 20 refund is issued with no approval.

## Step 3: Approve a refund that needs a person's decision

`decide` records the approver's decision and sends it back to the paused agent as a
function response with the same ID as the request. The tool runs only if the decision
is `confirmed: True`.

In [ ]:
def decide(session_id, request, approved, approver):
    """Sends the approver's decision back to the paused agent."""
    call = request.args["originalFunctionCall"]
    record("approval_decision", tool=call["name"], args=call["args"],
           approved=approved, approver=approver)
    print(f"   {approver} {'approved' if approved else 'rejected'} it.")
    response = types.Part(function_response=types.FunctionResponse(
        id=request.id, name=request.name,
        response={"confirmed": approved}))
    send(session_id, types.Content(role="user", parts=[response]))


first_session, first = ask("I need a refund of USD 400 for order 10422.")
decide(first_session, first[0], approved=True, approver="associate-ana")

**Expected result:** the agent pauses, `associate-ana approved it`, and the refund is issued.

## Step 4: Reject a refund, and test the guardrail

Reject a refund, and then request one that exceeds the order total.

In [ ]:
session_id, requests = ask("Refund USD 300 for order 10423, please.")
decide(session_id, requests[0], approved=False, approver="associate-raj")
ask("Refund USD 5000 for order 10423.")  # More than the order total.

**Expected result:** the rejected refund is never issued, and the USD 5000 refund is
blocked by the guardrail before anyone is asked to approve it.

## Step 5: Review the audit trail

Review the audit trail.

In [ ]:
for entry in AUDIT:
    details = {k: v for k, v in entry.items() if k not in ("time", "event")}
    print(f"{entry['time']}  {entry['event']:<19}{details}")

**Expected result:** every refund, decision, approver, and block, with a time.

## Step 6: Experiment

Run the next cell. It replays the approval request from Step 3 in the USD 300 conversation
from Step 4. ADK raises a `ValueError` because no paused call in that conversation matches
it, so an approval can't be reused to authorize a different action.

In [ ]:
try:
    decide(session_id, first[0], approved=True, approver="mallory")
except ValueError as error:
    print("Rejected by ADK:", str(error)[:90])
print("Refunds issued:", sum(e["event"] == "refund_issued" for e in AUDIT))

## Check Your Understanding

**1. Why is the order-total limit a callback rather than an approval?**

<details><summary>Show answer</summary>

It must never be broken, so it is enforced in code. Approvals are for decisions that need a person's judgment.

</details>

**2. What does ADK do when a tool needs confirmation?**

<details><summary>Show answer</summary>

It pauses the agent and emits an adk_request_confirmation function call. The tool runs only after the application sends back a function response with confirmed set to true.

</details>

**3. What should an approval record contain?**

<details><summary>Show answer</summary>

The proposed action and its parameters, the approver, the decision, and the time.

</details>

## Summary

You required human approval for large refunds, enforced a hard limit in code, and recorded
every decision. This lab created no resources, so there is nothing to clean up.